# Segment Effects

Tests the six segment hypotheses in the [analysis plan](../analysis_plan.md). The results feed the targeting rule for challenge questions 3 and 4, and H2 also answers questions 5 and 8.

Each hypothesis is tested with one regression that includes an interaction term. **The interaction coefficient is the difference in the email effect between the two segments.** Standard errors are robust (HC3). The Holm correction is applied after all six tests are run.

## How to read the tables

Each hypothesis fits `outcome = b0 + b1 × email + b2 × segment + b3 × email × segment`, where `segment` is 0 or 1. Each hypothesis shows two tables.

**Table 1, the raw regression output for spend:**

| Coefficient | Meaning |
| --- | --- |
| b0 | Mean spend in segment 0 without email. It is a baseline, and its p-value is not of interest. |
| b1 | Email effect in segment 0 |
| b2 | Difference between the segments without email. It describes the customers, not the email. |
| **b3** | **Difference in the email effect between the segments. Its p-value decides the hypothesis.** |

**Table 2, the effect in each segment, for all three metrics:**

- The first row is b1, and the third row is b3. Both are copied from the regression, with the same values and p-values as Table 1.
- The second row is b1 + b3, the email effect in segment 1. **It is not a regression coefficient, so it is computed separately** with a test of the sum: the estimate is b1 + b3, and its standard error comes from Var(b1) + Var(b3) + 2 × Cov(b1, b3). Because b3 is defined relative to b1, their errors cancel, and **the result equals comparing email and no email within segment 1 alone.** Its p-value therefore cannot be read from the p-values of b1 and b3.

**Each result note follows the same order:** the difference in effect (b3), which decides the hypothesis; the effect in each segment (b1 and b1 + b3); the difference without email (b2); the secondary metrics; and a one-sentence summary.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("../data/hillstrom.csv")
df["email"] = (df["segment"] != "No E-Mail").astype(int)

METRICS = ["spend", "conversion", "visit"]


def raw_coefficients(data, treatment, segment, labels, metric="spend"):
    """Fit outcome ~ treatment * segment and return the four regression coefficients with their meaning."""
    model = smf.ols(f"{metric} ~ {treatment} * {segment}", data).fit(cov_type="HC3")
    ci = model.conf_int()
    meaning = {
        "Intercept": f"b0: mean {metric} for {labels[0]}, no email",
        treatment: f"b1: email effect for {labels[0]}",
        segment: f"b2: difference without email, {labels[1]} minus {labels[0]}",
        f"{treatment}:{segment}": "b3: difference in email effect (interaction)",
    }
    table = pd.DataFrame({"value": model.params, "low": ci[0], "high": ci[1], "p_value": model.pvalues})
    return table.rename(index=meaning)


def interaction_test(data, treatment, segment, labels):
    """Fit outcome ~ treatment * segment for each metric and report the effect in each segment and their difference.

    labels maps the segment values 0 and 1 to readable names.
    """
    rows = []
    for metric in METRICS:
        model = smf.ols(f"{metric} ~ {treatment} * {segment}", data).fit(cov_type="HC3")
        tests = {
            f"effect for {labels[0]}": f"{treatment} = 0",
            f"effect for {labels[1]}": f"{treatment} + {treatment}:{segment} = 0",
            "difference (interaction)": f"{treatment}:{segment} = 0",
        }
        for label, hypothesis in tests.items():
            test = model.t_test(hypothesis)
            low, high = test.conf_int(alpha=0.05)[0]
            rows.append({"metric": metric, "estimate": label, "value": float(test.effect[0]),
                         "low": low, "high": high, "p_value": float(test.pvalue)})
    return pd.DataFrame(rows).set_index(["metric", "estimate"])

## H1: Recency

**Hypothesis:** the email effect is larger for customers who last purchased 1 to 3 months ago than for those who last purchased 4 to 12 months ago. The Mens and Womens groups are pooled.

Model: `spend ~ email + recent + email:recent`, where `recent` = 1 for 1 to 3 months.

In [2]:
df["recent"] = (df["recency"] <= 3).astype(int)
H1_LABELS = {0: "4-12 months (recent = 0)", 1: "1-3 months (recent = 1)"}

counts = pd.crosstab(df["recent"].map(H1_LABELS), df["email"].map({0: "No email", 1: "Email"}), margins=True)
counts.index.name, counts.columns.name = "Last purchase", None
counts

,Email,No email,All
Last purchase,,,
1-3 months (recent = 1),14955,7438,22393
4-12 months (recent = 0),27739,13868,41607
All,42694,21306,64000


In [3]:
display(raw_coefficients(df, "email", "recent", H1_LABELS).round(4))

h1 = interaction_test(df, "email", "recent", H1_LABELS)
h1.round(4)

,value,low,high,p_value
"b0: mean spend for 4-12 months (recent = 0), no email",0.4968,0.3223,0.6714,0.0000
b1: email effect for 4-12 months (recent = 0),0.5389,0.2923,0.7855,0.0000
"b2: difference without email, 1-3 months (recent = 1) minus 4-12 months (recent = 0)",0.4467,0.0957,0.7977,0.0126
b3: difference in email effect (interaction),0.1638,-0.3346,0.6622,0.5194


value     low    high  \
metric     estimate                                                      
spend      effect for 4-12 months (recent = 0)  0.5389  0.2923  0.7855   
           effect for 1-3 months (recent = 1)   0.7027  0.2696  1.1358   
           difference (interaction)             0.1638 -0.3346  0.6622   
conversion effect for 4-12 months (recent = 0)  0.0049  0.0034  0.0064   
           effect for 1-3 months (recent = 1)   0.0050  0.0022  0.0079   
           difference (interaction)             0.0001 -0.0031  0.0034   
visit      effect for 4-12 months (recent = 0)  0.0589  0.0525  0.0652   
           effect for 1-3 months (recent = 1)   0.0645  0.0543  0.0746   
           difference (interaction)             0.0056 -0.0063  0.0176   

                                                p_value  
metric     estimate                                      
spend      effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0015  
           difference (interaction)              0.5194  
conversion effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0006  
           difference (interaction)              0.9363  
visit      effect for 4-12 months (recent = 0)   0.0000  
           effect for 1-3 months (recent = 1)    0.0000  
           difference (interaction)              0.3578

**Reading the result (spend):**

- **Difference in effect (b3): +$0.16, 95% CI -$0.33 to $0.66, p = 0.52. H1 is not supported:** the data cannot show that the email worked differently for recent customers.
- **Effect in each group (b1 and b1 + b3):** the email raised spend for both, by +$0.54 for customers who last purchased 4-12 months ago and +$0.70 for those who purchased 1-3 months ago.
- **Difference without email (b2):** recent customers spent $0.45 more even without email (p = 0.013). This describes the customers, not the email.
- **Secondary metrics:** conversion and visit lead to the same conclusion.

**Summary: the email worked for both recent and less recent customers, by similar amounts.** The Holm correction is applied after all six tests.

## H2a: Product match for the Womens email

**Hypothesis:** the Womens email effect is larger for customers who bought women's merchandise than for those who bought only men's.

Only the Womens E-Mail and No E-Mail groups are used, because the question is about the Womens email alone. Including the Mens group would mix in the effect of a different email. Model: `spend ~ womens_email + womens + womens_email:womens`.

In [4]:
womens_data = df[df["segment"] != "Mens E-Mail"].assign(womens_email=lambda frame: (frame["segment"] == "Womens E-Mail").astype(int))
H2A_LABELS = {0: "bought only men's (womens = 0)", 1: "bought women's (womens = 1)"}

counts = pd.crosstab(womens_data["womens"].map(H2A_LABELS), womens_data["womens_email"].map({0: "No email", 1: "Womens email"}), margins=True)
counts.index.name, counts.columns.name = "Past purchases", None
display(counts)

display(raw_coefficients(womens_data, "womens_email", "womens", H2A_LABELS).round(4))

h2a = interaction_test(womens_data, "womens_email", "womens", H2A_LABELS)
h2a.round(4)

,No email,Womens email,All
Past purchases,,,
bought only men's (womens = 0),9638,9622,19260
bought women's (womens = 1),11668,11765,23433
All,21306,21387,42693


,value,low,high,p_value
"b0: mean spend for bought only men's (womens = 0), no email",0.6954,0.4460,0.9448,0.0000
b1: email effect for bought only men's (womens = 0),0.2784,-0.1236,0.6803,0.1747
"b2: difference without email, bought women's (womens = 1) minus bought only men's (womens = 0)",-0.0779,-0.3949,0.2392,0.6303
b3: difference in email effect (interaction),0.2659,-0.2528,0.7845,0.3151


value     low    high  \
metric     estimate                                                            
spend      effect for bought only men's (womens = 0)  0.2784 -0.1236  0.6803   
           effect for bought women's (womens = 1)     0.5442  0.2164  0.8721   
           difference (interaction)                   0.2659 -0.2528  0.7845   
conversion effect for bought only men's (womens = 0)  0.0006 -0.0015  0.0028   
           effect for bought women's (womens = 1)     0.0051  0.0028  0.0075   
           difference (interaction)                   0.0045  0.0013  0.0077   
visit      effect for bought only men's (womens = 0)  0.0111  0.0024  0.0197   
           effect for bought women's (womens = 1)     0.0731  0.0641  0.0822   
           difference (interaction)                   0.0620  0.0495  0.0746   

                                                      p_value  
metric     estimate                                            
spend      effect for bought only men's (womens = 0)   0.1747  
           effect for bought women's (womens = 1)      0.0011  
           difference (interaction)                    0.3151  
conversion effect for bought only men's (womens = 0)   0.5637  
           effect for bought women's (womens = 1)      0.0000  
           difference (interaction)                    0.0055  
visit      effect for bought only men's (womens = 0)   0.0123  
           effect for bought women's (womens = 1)      0.0000  
           difference (interaction)                    0.0000

**Reading the result (spend):**

- **Difference in effect (b3): +$0.27, 95% CI -$0.25 to $0.78, p = 0.32. H2a is not supported on spend.**
- **Effect in each group (b1 and b1 + b3):** the Womens email added +$0.54 for customers who bought women's merchandise (p = 0.001) and +$0.28 for those who bought only men's (p = 0.17). One effect is significant and the other is not, but because the difference itself is not significant, this does not show that the email worked only for women's buyers.
- **Difference without email (b2):** no detectable difference between the groups (-$0.08, p = 0.63).
- **Secondary metrics:** the difference in effect is significant for conversion (+0.45 pp, p = 0.006) and visit (+6.2 pp, p < 0.001). As set out in the plan, secondary metrics are not used to claim significance.

**Summary: the Womens email draws past women's buyers to visit and buy more often, but the difference in spend is too uncertain to confirm.** The Holm correction is applied after all six tests.

## H2b: Product match for the Mens email

**Hypothesis:** the Mens email effect is larger for customers who bought men's merchandise than for those who bought only women's.

Only the Mens E-Mail and No E-Mail groups are used. Model: `spend ~ mens_email + mens + mens_email:mens`.

In [5]:
mens_data = df[df["segment"] != "Womens E-Mail"].assign(mens_email=lambda frame: (frame["segment"] == "Mens E-Mail").astype(int))
H2B_LABELS = {0: "bought only women's (mens = 0)", 1: "bought men's (mens = 1)"}

counts = pd.crosstab(mens_data["mens"].map(H2B_LABELS), mens_data["mens_email"].map({0: "No email", 1: "Mens email"}), margins=True)
counts.index.name, counts.columns.name = "Past purchases", None
display(counts)

display(raw_coefficients(mens_data, "mens_email", "mens", H2B_LABELS).round(4))

h2b = interaction_test(mens_data, "mens_email", "mens", H2B_LABELS)
h2b.round(4)

,Mens email,No email,All
Past purchases,,,
bought men's (mens = 1),11739,11787,23526
bought only women's (mens = 0),9568,9519,19087
All,21307,21306,42613


,value,low,high,p_value
"b0: mean spend for bought only women's (mens = 0), no email",0.4906,0.3088,0.6723,0.0000
b1: email effect for bought only women's (mens = 0),0.6157,0.2697,0.9618,0.0005
"b2: difference without email, bought men's (mens = 1) minus bought only women's (mens = 0)",0.2932,-0.0078,0.5942,0.0562
b3: difference in email effect (interaction),0.2809,-0.2731,0.8349,0.3203


value     low    high  \
metric     estimate                                                            
spend      effect for bought only women's (mens = 0)  0.6157  0.2697  0.9618   
           effect for bought men's (mens = 1)         0.8966  0.4640  1.3292   
           difference (interaction)                   0.2809 -0.2731  0.8349   
conversion effect for bought only women's (mens = 0)  0.0057  0.0032  0.0082   
           effect for bought men's (mens = 1)         0.0077  0.0051  0.0102   
           difference (interaction)                   0.0020 -0.0016  0.0056   
visit      effect for bought only women's (mens = 0)  0.0706  0.0611  0.0801   
           effect for bought men's (mens = 1)         0.0816  0.0724  0.0908   
           difference (interaction)                   0.0110 -0.0023  0.0242   

                                                      p_value  
metric     estimate                                            
spend      effect for bought only women's (mens = 0)   0.0005  
           effect for bought men's (mens = 1)          0.0000  
           difference (interaction)                    0.3203  
conversion effect for bought only women's (mens = 0)   0.0000  
           effect for bought men's (mens = 1)          0.0000  
           difference (interaction)                    0.2818  
visit      effect for bought only women's (mens = 0)   0.0000  
           effect for bought men's (mens = 1)          0.0000  
           difference (interaction)                    0.1039

**Reading the result (spend):**

- **Difference in effect (b3): +$0.28, 95% CI -$0.27 to $0.83, p = 0.32. H2b is not supported.**
- **Effect in each group (b1 and b1 + b3):** the Mens email raised spend for both, by +$0.62 for customers who bought only women's merchandise and +$0.90 for those who bought men's. Both effects are significant.
- **Difference without email (b2):** men's buyers spent $0.29 more without email, but this is not significant (p = 0.056).
- **Secondary metrics:** the difference in effect is not significant for conversion or visit either.

**Summary: the Mens email worked for both groups, with no evidence that it worked better for men's buyers.** In contrast, the Womens email's extra visits and purchases were concentrated among women's buyers. This contrast compares two separate tests and was not tested directly, so it is descriptive only. The Holm correction is applied after all six tests.

## H3: Channel

**Hypothesis:** the email effect is larger for Web and Multichannel customers than for Phone customers. The Mens and Womens groups are pooled.

Model: `spend ~ email + online + email:online`, where `online` = 1 for Web or Multichannel and 0 for Phone.

In [6]:
df["online"] = (df["channel"] != "Phone").astype(int)
H3_LABELS = {0: "Phone (online = 0)", 1: "Web or Multichannel (online = 1)"}

counts = pd.crosstab(df["channel"], df["email"].map({0: "No email", 1: "Email"}), margins=True)
counts.index.name, counts.columns.name = "Channel", None
display(counts)

display(raw_coefficients(df, "email", "online", H3_LABELS).round(4))

h3 = interaction_test(df, "email", "online", H3_LABELS)
h3.round(4)

,Email,No email,All
Channel,,,
Multichannel,5156,2606,7762
Phone,18694,9327,28021
Web,18844,9373,28217
All,42694,21306,64000


,value,low,high,p_value
"b0: mean spend for Phone (online = 0), no email",0.6445,0.4073,0.8816,0.0000
b1: email effect for Phone (online = 0),0.3964,0.0811,0.7117,0.0137
"b2: difference without email, Web or Multichannel (online = 1) minus Phone (online = 0)",0.0148,-0.2994,0.3291,0.9263
b3: difference in email effect (interaction),0.3565,-0.0829,0.7960,0.1118


value     low  \
metric     estimate                                                      
spend      effect for Phone (online = 0)                0.3964  0.0811   
           effect for Web or Multichannel (online = 1)  0.7529  0.4468   
           difference (interaction)                     0.3565 -0.0829   
conversion effect for Phone (online = 0)                0.0036  0.0016   
           effect for Web or Multichannel (online = 1)  0.0060  0.0041   
           difference (interaction)                     0.0025 -0.0003   
visit      effect for Phone (online = 0)                0.0599  0.0523   
           effect for Web or Multichannel (online = 1)  0.0616  0.0540   
           difference (interaction)                     0.0017 -0.0091   

                                                          high  p_value  
metric     estimate                                                      
spend      effect for Phone (online = 0)                0.7117   0.0137  
           effect for Web or Multichannel (online = 1)  1.0590   0.0000  
           difference (interaction)                     0.7960   0.1118  
conversion effect for Phone (online = 0)                0.0056   0.0005  
           effect for Web or Multichannel (online = 1)  0.0080   0.0000  
           difference (interaction)                     0.0053   0.0852  
visit      effect for Phone (online = 0)                0.0676   0.0000  
           effect for Web or Multichannel (online = 1)  0.0692   0.0000  
           difference (interaction)                     0.0125   0.7601

**Reading the result (spend):**

- **Difference in effect (b3): +$0.36, 95% CI -$0.08 to $0.80, p = 0.11. H3 is not supported.**
- **Effect in each group (b1 and b1 + b3):** the email raised spend for both, by +$0.40 for Phone customers (p = 0.014) and +$0.75 for Web or Multichannel customers (p < 0.001). The second estimate is almost twice the first, but the difference is estimated too imprecisely to rule out chance.
- **Difference without email (b2):** no detectable difference between the groups (+$0.01, p = 0.93).
- **Secondary metrics:** the difference in effect is not significant for conversion (+0.25 pp, p = 0.085) or visit (+0.2 pp, p = 0.76). The visit effect is almost the same in both groups (+6.0 pp and +6.2 pp), so Phone customers also visited the website after the email.

**Summary: the email worked for both Phone and online customers, with no clear evidence of a difference.** The Holm correction is applied after all six tests.